In [8]:
%load_ext autoreload
%autoreload 2


import numpy as np
import matplotlib
matplotlib.use('Agg')
import warnings
import pandas as pd
from datetime import datetime
from matplotlib import pyplot as plt
import seaborn as sns

#Set random seeds
seed = 210226
warnings.filterwarnings('ignore', category=RuntimeWarning)

sns.set_theme(style="white", font_scale=1.2)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Set BCH-specific variables.

In [6]:
#####-------FILENAMES ETC--------------
dataset_name = "bch" 
raw_file_name = "preprocessed-visits-with-raw-variables"
load_filepath = "/lab-share/CHIP-Lacava-e2/Groups/BCH-ED/epi-and-prediction/"
save_filepath = "/lab-share/CHIP-Lacava-e2/Groups/BCH-ED/epi-and-prediction/prediction/recent-eval/at-triage/"
plot_label = "at triage"

raw_filepath = "/lab-share/CHIP-Lacava-e2/Groups/BCH-ED/raw-data/"


####--------PREPROCESSING VARIABLES------------

#Define identifier columns.
identifier_cols = ["csn", "is_admitted", "mrn", "minutes_since_first_arrival", "raw_arrival_month", "raw_arrival_year"]

#Define excluded columns-- we do not use these as predictors.
excluded_columns = identifier_cols + ["group", "assigned_order"] #These are excluded from both predictions and IPW calculations.
excluded_prefixes = ['current_diagnosis'] #These are excluded from just predictions.

#Identify gendered predictions; we exclude these when matching patients across sex-based groups
gendered_predictors = ['current_diagnosis_menstrual_disorders', 'pre_diagnosis_menstrual_disorders', 'complaint_contains_abdominal_pain',
                       'complaint_contains_gynecologic', 'complaint_contains_male_genital', 'complaint_contains_pregnancy']

#Separate categorical, pre-binarised, and continuous predictors.
categorical_predictors = [
    'race', 'sex', 'ed_arrival_mode', 'language',
                'insurance', 'home_state', 'year_of_arrival', 'season', 'time_of_day', 'triage_acuity']



#Variables which already 'come pre-binarised'.
pre_binarised_predictors = ['is_trans_or_nb', 'is_weekend',
                            'current_diagnosis_epilepsy', 'current_diagnosis_asthma', 'current_diagnosis_congenital_malformations', 
                'current_diagnosis_depression', 'current_diagnosis_pain_conditions', 'current_diagnosis_anxiety',
                  'current_diagnosis_conduct_disorders', 'current_diagnosis_nausea_and_vomiting',
                    'current_diagnosis_any_malignancy', 'current_diagnosis_gastrointestinal', 
                    'current_diagnosis_developmental_delays', 'current_diagnosis_psychotic_disorders', 
                    'current_diagnosis_cardiovascular', 'current_diagnosis_diabetes_mellitus', 'current_diagnosis_eating_disorders', 
                    'current_diagnosis_weight_loss', 'current_diagnosis_anemia', 'current_diagnosis_chromosomal_anomalies', 
                    'current_diagnosis_drug_abuse', 'current_diagnosis_menstrual_disorders', 'current_diagnosis_sleep_disorders', 
                    'current_diagnosis_smoking', 'current_diagnosis_alcohol_abuse', 'current_diagnosis_joint_disorders', 
                      'pre_diagnosis_any_malignancy', 'pre_diagnosis_gastrointestinal', 'pre_diagnosis_nausea_and_vomiting', 
                      'pre_diagnosis_diabetes_mellitus', 'pre_diagnosis_pain_conditions', 'pre_diagnosis_cardiovascular',
                        'pre_diagnosis_developmental_delays', 'pre_diagnosis_epilepsy', 'pre_diagnosis_asthma', 'pre_diagnosis_anemia', 
                        'pre_diagnosis_congenital_malformations', 'pre_diagnosis_conduct_disorders', 'pre_diagnosis_chromosomal_anomalies',
                          'pre_diagnosis_anxiety', 'pre_diagnosis_weight_loss', 'pre_diagnosis_psychotic_disorders', 'pre_diagnosis_drug_abuse',
                            'pre_diagnosis_smoking', 'pre_diagnosis_depression', 'pre_diagnosis_eating_disorders', 'pre_diagnosis_menstrual_disorders', 
                            'pre_diagnosis_sleep_disorders', 'pre_diagnosis_joint_disorders', 'pre_diagnosis_alcohol_abuse', 
                            'complaint_contains_abdominal_pain', 'complaint_contains_assault', 
                            'complaint_contains_allergic_reaction', 'complaint_contains_altered_mental_status',
                              'complaint_contains_asthma_or_wheezing', 'complaint_contains_bites_or_stings', 
                              'complaint_contains_burn', 'complaint_contains_cardiac', 'complaint_contains_chest_pain', 
                              'complaint_contains_chronic_disease', 'complaint_contains_congestion', 'complaint_contains_constipation', 
                              'complaint_contains_cough', 'complaint_contains_croup', 'complaint_contains_crying_or_colic', 'complaint_contains_dental', 
                              'complaint_contains_device_complication', 'complaint_contains_diarrhea', 'complaint_contains_ear_complaint',
                                'complaint_contains_epistaxis', 'complaint_contains_extremity', 'complaint_contains_eye_complaint', 
                                'complaint_contains_syncope', 'complaint_contains_foreign_body', 'complaint_contains_fever', 
                                'complaint_contains_follow_up', 'complaint_contains_general', 'complaint_contains_gi_bleed',
                                  'complaint_contains_gynecologic', 'complaint_contains_head_or_neck', 'complaint_contains_headache', 
                                  'complaint_contains_laceration', 'complaint_contains_lump_or_mass', 'complaint_contains_male_genital', 
                                  'complaint_contains_mvc', 'complaint_contains_neck_pain', 'complaint_contains_neurologic', 
                                  'complaint_contains_poisoning', 'complaint_contains_poor_feeding', 'complaint_contains_pregnancy', 
                                  'complaint_contains_primary_care', 'complaint_contains_psych', 'complaint_contains_rash', 
                                  'complaint_contains_other_respiratory', 'complaint_contains_seizure', 'complaint_contains_sore_throat',
                                    'complaint_contains_trauma', 'complaint_contains_urinary', 'complaint_contains_vomiting',
                    ]


#Continuous predictors.
continuous_predictors = [
    'age_in_days', 'miles_travelled', 'sdi_score',
    'num_previous_admissions',
    'num_previous_visits_without_admission',
    'raw_triage_hr', 'raw_triage_rr', 'raw_triage_sp_o2', 'raw_triage_pain', 'raw_triage_sbp', 
    'raw_triage_dbp',
     'crowdedness', 'pseudo_nedocs', 'raw_weight'
    
]


#First, define the reference categories.
reference_categories = [
    "Non-Hispanic White", #race
    "M", #sex
    "Walk in", #ed arrival mode
    "English", #language
    "Private", #insurance
    "MA", #home state
    "2019", #year of arrival
    "autumn", #season
    "afternoon", #time of day,
    "unknown" #triage acuity
]
#Map races into groups.
race_dict = {
    'Hispanic': 'H',
    'Hispanic White': 'H',
    'Black': 'NHB',
    'Non-Hispanic Black': 'NHB',
    'Asian' : 'Other',
    'Unknown': 'Other',
    'Other': 'Other',
    'White': 'NHW',
    'Non-Hispanic White' : 'NHW'
}

#Name demographic categories.
sex_name = 'sex'
race_name = 'race'

####----TRAINING SETTINGS------
test_size = 6 #Number of months to use as test set
time_span_of_validation_set = 12 # we always use the first year for training/testing
update_intervals = [0.5] #Update every 2 weeks
final_update_interval = 0.5
frac_test_set_for_evaluation = 0.5 #evaluate hyperparameters on second half of the test set.
in_order=True

###------IPW SETTINGS--------
ipw_load_path = None #If none, IPWs are calculated; if a path exists we load it in from somewhere else

###-----OUTCOMES----------
false_admission_outcomes = {
    'dichotomous': ['death_in_hospital'],
    'continuous': {'hosp_los':['death_in_hospital']}, #Key-value pairs are time events, censoring events
}

false_discharge_outcomes = {
    'dichotomous': ['visit_within_30_days', 'visit_within_1_week', 'readmission_within_30_days', 'readmission_within_1_week'],
    'continuous': {},
}

In [ ]:
#Clean variable names.
def clean_names(df):
    df = df.copy()
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_', regex=False)
                  .str.replace(r'[^\w\s]', '_', regex=True)
                  .str.replace('_+', '_', regex=True)
                  .str.strip('_'))
    return df

---PRE-PREPROCESSING STEP---

Link continuous variables (age, triage vitals) to the epi-preprocessed dataset.

In [ ]:
#Load in the data-- this is the full cohort we want to use.
visits = pd.read_csv(load_filepath+"preprocessed-visits.csv", index_col=0, engine='python') #339400 visits


print("raw visits loaded")
#Link month and year of arrival, raw weight, and state of origin
arrival_times = pd.read_csv(load_filepath + "intermediate-files/visits-with-linked-demographics.csv")[['csn', 'arrival_time', 'weight', 'mrn']].rename(columns={
    'weight':'raw_weight'
})

print("arrival times loaded")
# state_of_origin = pd.read_csv(load_filepath + "intermediate-files/visits-with-sdi-scores.csv", index_col=0, engine='python')[['csn', 'home_state']]
visits = visits.merge(arrival_times, on="csn", how="inner")#.merge(state_of_origin, on="csn", how="left")

#Attach arrival month and year
visits['raw_arrival_month'] = pd.to_datetime(visits['arrival_time'], errors = 'coerce').dt.month
visits['raw_arrival_year'] = pd.to_datetime(visits['arrival_time'], errors = 'coerce').dt.year

earliest_arrival = pd.to_datetime(visits['arrival_time']).min()
visits['minutes_since_first_arrival'] = pd.to_datetime(visits['arrival_time']).apply(lambda t: (t-earliest_arrival).total_seconds()/60)

#LOOK ONLY AT PRE-EPIC-TRANSITION VISITS (May 2024 and earlier).
pre_epic_flag = pd.read_csv(load_filepath+"preprocessed-visits-with-linked-events.csv", index_col=0)[['csn', 'pre_epic']]
visits = visits.merge(pre_epic_flag, how='inner', on='csn')
visits = visits[visits['pre_epic']==1].drop(columns=['pre_epic'])

#Save the covariates we need.
cols_to_keep = ['csn', 'race', 'sex', 'ed_arrival_mode', 'language', 
                'insurance', 'age_in_days', 'is_admitted', 'is_trans_or_nb', 
                'miles_travelled', 'home_state', 'sdi_score',
                'arbitrary_timestamp', 'num_previous_admissions', 'num_previous_visits_without_admission', 'year_of_arrival', 'season', 'is_weekend', 'time_of_day',
                'raw_triage_dbp', 'raw_triage_hr', 'raw_triage_pain', 'raw_triage_rr', 'raw_triage_sbp', 'raw_triage_sp_o2', 'raw_triage_pain',
                'current_diagnosis_epilepsy', 'current_diagnosis_asthma', 'current_diagnosis_congenital_malformations', 
                'current_diagnosis_depression', 'current_diagnosis_pain_conditions', 'current_diagnosis_anxiety',
                  'current_diagnosis_conduct_disorders', 'current_diagnosis_nausea_and_vomiting',
                    'current_diagnosis_any_malignancy', 'current_diagnosis_gastrointestinal', 
                    'current_diagnosis_developmental_delays', 'current_diagnosis_psychotic_disorders', 
                    'current_diagnosis_cardiovascular', 'current_diagnosis_diabetes_mellitus', 'current_diagnosis_eating_disorders', 
                    'current_diagnosis_weight_loss', 'current_diagnosis_anemia', 'current_diagnosis_chromosomal_anomalies', 
                    'current_diagnosis_drug_abuse', 'current_diagnosis_menstrual_disorders', 'current_diagnosis_sleep_disorders', 
                    'current_diagnosis_smoking', 'current_diagnosis_alcohol_abuse', 'current_diagnosis_joint_disorders', 
                      'pre_diagnosis_any_malignancy', 'pre_diagnosis_gastrointestinal', 'pre_diagnosis_nausea_and_vomiting', 
                      'pre_diagnosis_diabetes_mellitus', 'pre_diagnosis_pain_conditions', 'pre_diagnosis_cardiovascular',
                        'pre_diagnosis_developmental_delays', 'pre_diagnosis_epilepsy', 'pre_diagnosis_asthma', 'pre_diagnosis_anemia', 
                        'pre_diagnosis_congenital_malformations', 'pre_diagnosis_conduct_disorders', 'pre_diagnosis_chromosomal_anomalies',
                          'pre_diagnosis_anxiety', 'pre_diagnosis_weight_loss', 'pre_diagnosis_psychotic_disorders', 'pre_diagnosis_drug_abuse',
                            'pre_diagnosis_smoking', 'pre_diagnosis_depression', 'pre_diagnosis_eating_disorders', 'pre_diagnosis_menstrual_disorders', 
                            'pre_diagnosis_sleep_disorders', 'pre_diagnosis_joint_disorders', 'pre_diagnosis_alcohol_abuse', 'triage_acuity',
                            'crowdedness', 'pseudo_nedocs', 'complaint_contains_abdominal_pain', 'complaint_contains_assault', 
                            'complaint_contains_allergic_reaction', 'complaint_contains_altered_mental_status',
                              'complaint_contains_asthma_or_wheezing', 'complaint_contains_bites_or_stings', 
                              'complaint_contains_burn', 'complaint_contains_cardiac', 'complaint_contains_chest_pain', 
                              'complaint_contains_chronic_disease', 'complaint_contains_congestion', 'complaint_contains_constipation', 
                              'complaint_contains_cough', 'complaint_contains_croup', 'complaint_contains_crying_or_colic', 'complaint_contains_dental', 
                              'complaint_contains_device_complication', 'complaint_contains_diarrhea', 'complaint_contains_ear_complaint',
                                'complaint_contains_epistaxis', 'complaint_contains_extremity', 'complaint_contains_eye_complaint', 
                                'complaint_contains_syncope', 'complaint_contains_foreign_body', 'complaint_contains_fever', 
                                'complaint_contains_follow_up', 'complaint_contains_general', 'complaint_contains_gi_bleed',
                                  'complaint_contains_gynecologic', 'complaint_contains_head_or_neck', 'complaint_contains_headache', 
                                  'complaint_contains_laceration', 'complaint_contains_lump_or_mass', 'complaint_contains_male_genital', 
                                  'complaint_contains_mvc', 'complaint_contains_neck_pain', 'complaint_contains_neurologic', 
                                  'complaint_contains_poisoning', 'complaint_contains_poor_feeding', 'complaint_contains_pregnancy', 
                                  'complaint_contains_primary_care', 'complaint_contains_psych', 'complaint_contains_rash', 
                                  'complaint_contains_other_respiratory', 'complaint_contains_seizure', 'complaint_contains_sore_throat',
                                    'complaint_contains_trauma', 'complaint_contains_urinary', 'complaint_contains_vomiting', 'raw_weight',
                                     'raw_arrival_month', 'raw_arrival_year', 'minutes_since_first_arrival', 'mrn'
                ]

#Keep diagnosis, but remember these will only be used for weighting.
visits = clean_names(visits[cols_to_keep])


#Now save this.
visits.to_csv(load_filepath+raw_file_name+".csv")